In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("13_races.csv")
df = df.drop(columns=["Jurisdiction", "Unprocessed Ballots", "Unnamed: 0"]).drop_duplicates()
df = df.sort_values(by=["Batch_ID"])

In [3]:
df['New_Total'] = df.groupby('Race')['Total Votes Cast'].diff().fillna(df['Total Votes Cast'])
df['New_D'] = df.groupby('Race')['D'].diff().fillna(df['D'])

df['New_Dem_Prop'] = np.where(
    df['New_Total'] > 0, 
    df['New_D'] / df['New_Total'], 
    np.nan
)

df

,Date,Timestamp,R,D,Margin,Daily Margin Change,Vote Difference,Total Votes Cast,Total Unprocessed Ballots*,Race,Datetime,Time_Gap,Batch_ID,Dem_Share,Day_Index,Final_Total_Votes,Percent_of_Final,New_Total,New_D,New_Dem_Prop
0,2024-11-06,17:25:00,46987,44568,0.026421,0.014072,2419,91555,NaN,AD-58,2024-11-06 17:25:00,0 days 00:00:00,1,0.486789,1,155988,0.586936,91555.0,44568.0,0.486789
28,2024-11-06,17:25:00,56785,53596,0.028891,0.000000,3189,110381,NaN,CD-13,2024-11-06 17:25:00,0 days 00:00:00,1,0.485555,1,210921,0.523329,110381.0,53596.0,0.485555
56,2024-11-06,17:25:00,58428,59437,0.008561,0.000000,1009,117865,NaN,CD-21,2024-11-06 17:25:00,0 days 00:00:00,1,0.504280,1,195531,0.602794,117865.0,59437.0,0.504280
336,2024-11-06,17:25:00,90791,91382,0.003244,0.000000,591,182173,NaN,SD-5,2024-11-06 17:25:00,0 days 00:00:00,1,0.501622,1,358804,0.507723,182173.0,91382.0,0.501622
84,2024-11-06,17:25:00,53794,43955,0.100656,0.000000,9839,97749,NaN,CD-22,2024-11-06 17:25:00,0 days 00:00:00,1,0.449672,1,167507,0.583552,97749.0,43955.0,0.449672
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,2024-12-05,10:16:00,92733,102798,0.051475,0.000000,10065,195531,0.0,CD-21,2024-12-05 10:16:00,0 days 00:00:00,29,0.525738,28,195531,1.000000,0.0,0.0,NaN
251,2024-12-05,10:16:00,180950,197397,0.043471,0.000000,16447,378347,0.0,CD-49,2024-12-05 10:16:00,0 days 00:00:00,29,0.521735,28,378347,1.000000,0.0,0.0,NaN
223,2024-12-05,10:16:00,171554,181721,0.028779,0.000000,10167,353275,0.0,CD-47,2024-12-05 10:16:00,0 days 00:00:00,29,0.514390,28,353275,1.000000,0.0,0.0,NaN
55,2024-12-05,10:16:00,105367,105554,0.000887,0.000000,187,210921,0.0,CD-13,2024-12-05 10:16:00,0 days 00:00:00,29,0.500443,28,210921,1.000000,0.0,0.0,NaN


In [4]:
df_before_nov8 = df[df['Datetime'] < '2024-11-08']

final_totals_before_nov8 = df_before_nov8.groupby('Race').last().reset_index()

final_totals_before_nov8['Dem_Prop_Before_11_08'] = (
    final_totals_before_nov8['D'] / final_totals_before_nov8['Total Votes Cast']
)

summary_df = final_totals_before_nov8[['Race', 'Datetime', 'D', 'Total Votes Cast', 'Dem_Prop_Before_11_08']]
summary_df.to_csv("first_drop.csv")

In [ ]:
drops = df[df['Datetime'] >= '2024-11-08'].copy()
drops['Votes_Remaining'] = drops['Final_Total_Votes'] - drops['Total Votes Cast']
data = drops[["Race", "Batch_ID", "New_Dem_Prop", "New_Total", "Votes_Remaining"]]
data = data.dropna()
data.to_csv("later_drops.csv")